# Patient Clustering on Blood-Test Data (HCV Dataset) with K-Means

**Goal:** find out whether routine blood-test values alone can group patients into meaningful clusters, without using the diagnosis labels, and then check how well those clusters line up with the real diagnosis categories.

**Dataset:** [HCV data, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/571/hcv+data) (615 patients, 14 columns: age, sex, 10 lab values and a diagnosis category).

**Method:** clean and scale the data, run K-means for several values of K, choose K with similarity and silhouette scores, then profile and visualise the clusters.

**Tools:** Python, pandas, NumPy, scikit-learn, SciPy, matplotlib.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (7, 4.5)

## 2. Load the data

Download the dataset from the UCI link above and save it as `data/hcv_dataset.csv` in this repository.
If you run this notebook in Google Colab without the file, the cell below will ask you to upload it.

In [ ]:
import os

DATA_PATH = "data/hcv_dataset.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files  # only exists inside Colab
        print("Dataset not found. Please upload the HCV CSV file.")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put the dataset at {DATA_PATH} (see the link above).")

data = pd.read_csv(DATA_PATH)
print("Dataset shape:", data.shape)
data.head()

## 3. Clean the data

Two things to handle before clustering:

1. **The `Unnamed: 0` column is only a row number.** The file is sorted by diagnosis category, so the row number would leak label information into the clusters. It must not be used as a feature.
2. **Some lab values are missing.** I fill them with the column **median**, which is less affected by outliers than the mean.

In [ ]:
# Drop the row-number column if it exists
data = data.drop(columns=[c for c in data.columns if c.startswith("Unnamed")])

print("Missing values per column:")
print(data.isnull().sum()[data.isnull().sum() > 0])

feature_cols = data.select_dtypes(include=[np.number]).columns.tolist()
data[feature_cols] = data[feature_cols].fillna(data[feature_cols].median())

print("\nFeatures used for clustering:", feature_cols)
print("\nDiagnosis categories (not used for clustering, only for checking afterwards):")
print(data["Category"].value_counts())

## 4. Quick look at the features

`Sex` and `Category` are text columns, so only age and the 10 lab values are clustered.
`Category` is kept aside to evaluate the clusters at the end.

In [ ]:
data[feature_cols].hist(bins=25, figsize=(12, 8))
plt.tight_layout()
plt.show()

data[feature_cols].describe().round(2)

## 5. Scale the features

Lab values have very different ranges (for example ALP is in the hundreds, ALB around 40, CHOL around 5), so every feature is standardised to mean 0 and standard deviation 1. Otherwise the large-scale columns would dominate K-means.

In [ ]:
X = data[feature_cols].to_numpy()
X_scaled = StandardScaler().fit_transform(X)

## 6. Choose the number of clusters (K)

I compare K = 2 to 8 using four measures:

- **Inertia** (elbow method): lower is tighter, look for the point where improvement slows down.
- **Silhouette score:** higher is better (maximum 1).
- **Intra-cluster cosine similarity:** average similarity of points to their own centroid (higher is better).
- **Inter-cluster cosine similarity:** average similarity between different centroids (lower is better).

The "hybrid score" is *intra minus inter*, so a higher value means clusters that are tight and well separated.

In [ ]:
def fit_kmeans(k, X):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X)
    return model, labels


def cosine_cluster_quality(X, labels, centroids):
    """Average cosine similarity inside clusters and between centroids."""
    intra = [
        cosine_similarity(X[labels == i], centroids[i].reshape(1, -1)).mean()
        for i in range(len(centroids))
    ]
    inter = cosine_similarity(centroids)
    off_diagonal = inter[~np.eye(len(centroids), dtype=bool)]
    return float(np.mean(intra)), float(off_diagonal.mean())


results = {}
for k in range(2, 9):
    model, labels = fit_kmeans(k, X_scaled)
    intra, inter = cosine_cluster_quality(X_scaled, labels, model.cluster_centers_)
    results[k] = {
        "model": model,
        "labels": labels,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "intra_sim": intra,
        "inter_sim": inter,
        "hybrid": intra - inter,
    }

summary = pd.DataFrame(
    {k: {m: v[m] for m in ["inertia", "silhouette", "intra_sim", "inter_sim", "hybrid"]}
     for k, v in results.items()}
).T.round(4)
summary.index.name = "K"
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(summary.index, summary["inertia"], marker="o")
axes[0].set_title("Elbow method (inertia)")

axes[1].plot(summary.index, summary["silhouette"], marker="o", color="tab:green")
axes[1].set_title("Silhouette score")

axes[2].plot(summary.index, summary["hybrid"], marker="o", color="tab:red")
axes[2].set_title("Hybrid score (intra - inter similarity)")

for ax in axes:
    ax.set_xlabel("K")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Choosing the final K

I pick the K with the highest hybrid score, then sanity-check it against the elbow and silhouette plots above.
If the three measures disagree, say so in the conclusion and explain which one you trusted.

In [ ]:
best_k = int(summary["hybrid"].idxmax())
best_model = results[best_k]["model"]
labels = results[best_k]["labels"]
print(f"Selected K = {best_k}")
sizes = {int(k): int(v) for k, v in zip(*np.unique(labels, return_counts=True))}
print("Cluster sizes:", sizes)

## 7. How compact is each cluster?

For each cluster I measure the **cosine distance** of its points to the centroid:
the *maximum* shows how spread out the cluster is, and the *mean* shows the typical distance.

In [ ]:
from scipy.spatial.distance import cdist

centroids = best_model.cluster_centers_
rows = []
for i in range(best_k):
    d = cdist(X_scaled[labels == i], centroids[i].reshape(1, -1), metric="cosine")
    rows.append({"cluster": i, "size": int((labels == i).sum()),
                 "max_distance_to_centroid": d.max(), "mean_distance_to_centroid": d.mean()})

pd.DataFrame(rows).set_index("cluster").round(4)

## 8. Visualise the clusters

The data has 11 dimensions, so I use PCA to project it to 2D for plotting only. The clustering itself was done on all 11 features.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5.5))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=labels, cmap="viridis", s=18, alpha=0.8)
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
plt.title(f"K-means clusters (K={best_k}) projected with PCA")
plt.grid(alpha=0.3)
plt.show()

## 9. What defines each cluster?

In [ ]:
profiled = data.copy()
profiled["cluster"] = labels

profile = profiled.groupby("cluster")[feature_cols].mean().round(2)
profile.insert(0, "n_patients", profiled["cluster"].value_counts().sort_index())
profile

In [ ]:
# Same table as z-scores: shows which features are unusually high or low in each cluster
z_profile = pd.DataFrame(X_scaled, columns=feature_cols).groupby(labels).mean()

fig, ax = plt.subplots(figsize=(9, 3 + 0.5 * best_k))
im = ax.imshow(z_profile.to_numpy(), cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(feature_cols)), feature_cols, rotation=45)
ax.set_yticks(range(best_k), [f"Cluster {i}" for i in range(best_k)])
ax.set_title("Cluster profile (standardised mean per feature)")
plt.colorbar(im, ax=ax, label="z-score")
plt.tight_layout()
plt.show()

## 10. Do the clusters match the real diagnoses?

The diagnosis was never given to the model, so this is an honest check of whether the lab values alone separate the groups.
The **adjusted Rand index (ARI)** is 1 for a perfect match and about 0 for random grouping.

In [ ]:
pd.crosstab(profiled["cluster"], profiled["Category"])

In [ ]:
ari = adjusted_rand_score(profiled["Category"], labels)
print(f"Adjusted Rand Index vs. diagnosis category: {ari:.3f}")

## 11. Conclusions

> **TODO (write after running the notebook, 3-5 sentences in your own words):**
> - Which K was selected and do the elbow / silhouette plots agree?
> - What distinguishes each cluster (use the profile table and heatmap)?
> - How well do the clusters match the diagnosis categories (use the crosstab and ARI)?

**Limitations**

- The dataset is small (615 patients) and heavily imbalanced: most patients are healthy blood donors, so the rare disease stages have very few examples.
- K-means assumes roughly round, similarly sized clusters, which may not describe real patient groups well.
- This is an unsupervised exploration, not a diagnostic tool. It should not be used for medical decisions.

**Possible next steps:** try other algorithms (hierarchical clustering, DBSCAN, Gaussian mixture), include `Sex` as an encoded feature, or train a supervised classifier and compare.